In [1]:
import anndata as ad
import pandas as pd
import numpy as np
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix
import muon
import scarches as sca
from multigrate.data import organize_multiome_anndatas
import scanpy as sc
import scib_metrics
from typing import Optional
import os, sys
from scipy.sparse import csr_matrix, coo_matrix
import scipy
from scipy import sparse
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
import scib
import scib_metrics
from scib_metrics.benchmark import Benchmarker
from typing import Any, Callable, Optional, Union
from plottable import ColumnDefinition, Table
from plottable.cmap import normed_cmap
from plottable.plots import bar

import warnings
warnings.filterwarnings("ignore")

/home/zhouweige/anaconda3/envs/scib/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 captum (see https://github.com/pytorch/captum).


In [2]:
def read_RNA_ADT(RNA_path,ADT_path):
    # gene expression
    cell_names = pd.read_csv(RNA_path+'/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids'] 
    cell_names['cell_ids'] = cell_names['cell_ids'].str.replace('.','-')
    X = csr_matrix(mmread(RNA_path+'/matrix.mtx').T)
    gene_names = pd.read_csv(RNA_path+'/features.tsv', sep = '\t',  header=None, index_col=None) 
    gene_names.columns =  ['gene_ids'] 
    adata_RNA = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = gene_names.gene_ids))
    adata_RNA.var_names_make_unique()
    # ADT information
    adata_ADT = pd.read_csv(f'{ADT_path}/ADT.csv', index_col=0)
    adata_ADT.columns = adata_ADT.columns.str.replace('.', '-')
    # adata_ADT.index = adata_ADT.index.str.replace('.', '_')
    # adata_ADT.index = adata_ADT.index.str.replace('-', '_')
    adata_ADT = ad.AnnData(adata_ADT.T)
    adata_ADT.X = adata_ADT.X.astype(np.float64)
    # numpy 转为 sparse matrix
    adata_ADT.X = csr_matrix(adata_ADT.X)
    
    return adata_RNA, adata_ADT

In [3]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def pca(adata, use_reps=None, n_comps=10):

    """Dimension reduction with PCA algorithm"""

    from sklearn.decomposition import PCA
    from scipy.sparse.csc import csc_matrix
    from scipy.sparse.csr import csr_matrix
    pca = PCA(n_components=n_comps)
    if use_reps is not None:
       feat_pca = pca.fit_transform(adata.obsm[use_reps])
    else:
       if isinstance(adata.X, csc_matrix) or isinstance(adata.X, csr_matrix):
          feat_pca = pca.fit_transform(adata.X.toarray())
       else:
          feat_pca = pca.fit_transform(adata.X)

    return feat_pca

def supervised_clustering(adata, n_clusters=7, key='Garfield', add_key='Garfield_cluster', cluster_method='leiden',
               start=0.5, end=2.0, increment=0.05, use_pca=False, n_comps=20):
    if use_pca:
       adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)

    method_cluster = add_key
    method = key
    
    if cluster_method == 'leiden':
       if use_pca:
          res = search_res(adata, n_clusters, use_rep=key + '_pca', cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       else:
          res = search_res(adata, n_clusters, use_rep=method, cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=0, key_added=method_cluster, resolution=res, neighbors_key=method)
    elif cluster_method == 'louvain':
       if use_pca:
          res = search_res(adata, n_clusters, use_rep=key + '_pca', cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       else:
          res = search_res(adata, n_clusters, use_rep=method, cluster_method=cluster_method, 
                           start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=0, key_added=method_cluster, resolution=res, neighbors_key=method)

def search_res(adata, n_clusters, use_rep='Garfield', cluster_method='leiden', start=0.5, end=2.0,
               increment=0.05, expansion_factor=1.75, max_iter=50):
    '''\
    Searching corresponding resolution according to given cluster number

    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Target number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float
        The end value for searching.
    increment : float
        The step size to increase.
    expansion_factor : float
        Factor to expand the search range if no resolution is found in the initial range.
    max_iter : int
        Maximum number of iterations to prevent infinite loops.

    Returns
    -------
    res : float
        Resolution.

    '''
    print('Searching resolution...')
    label = 0
    method = use_rep
    method_cluster = method + '_cluster'
    
    current_start, current_end = start, end
    best_res = None
    best_diff = float('inf')  # Track the best resolution so far
    best_count = 0
    iteration = 0  # Track number of iterations
    
    # 构图
    sc.pp.neighbors(adata, use_rep=use_rep, key_added=method)

    while label == 0 and iteration < max_iter:
        iteration += 1
        for res in sorted(list(np.arange(current_start, current_end, increment)), reverse=True):
            if cluster_method == 'leiden':
                sc.tl.leiden(adata, random_state=0, key_added=method_cluster,
                             resolution=res, neighbors_key=method)
                count_unique = adata.obs[method_cluster].nunique()
            elif cluster_method == 'louvain':
                sc.tl.louvain(adata,  random_state=0, key_added=method_cluster, 
                             resolution=method_reso, neighbors_key=method)
                count_unique = adata.obs[method_cluster].nunique()

            print(f'resolution={res}, cluster number={count_unique}')

            # Update the best resolution if it's closer to the target
            diff = abs(count_unique - n_clusters)
            if diff < best_diff:
                best_diff = diff
                best_res = res
                best_count = count_unique

            if count_unique == n_clusters:
                label = 1
                break

        if label == 0:
            if best_res is not None:
                # Instead of starting from scratch, continue from the closest found resolution
                if best_count > n_clusters:
                    current_start = min(best_res, current_start * 0.9)  # Avoid too small a search range
                    current_end = min(current_start * expansion_factor, end * 2)  # Limit max range
                    print(f"Expanding search range: new range ({current_start}, {current_end})")
                else:
                    current_start = max(best_res, current_start * 1.2)  # Avoid too small a search range
                    current_end = min(current_start * expansion_factor, end * 2)  # Limit max range
                    print(f"Expanding search range: new range ({current_start}, {current_end})")
            else:
                print("Warning: No valid resolution found yet, expanding the search range.")
                current_start *= expansion_factor
                current_end *= expansion_factor

    if iteration >= max_iter:
        print("Warning: Reached maximum iteration limit.")

    return best_res if label == 1 else None

In [4]:
def count_metrics_for_resolution(dataset, reso_dict=None, start=0.5, end=2.0, increment=0.05):
    rna_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_Protein/{dataset}/RNA'
    adt_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_Protein/{dataset}'
    metadata_file = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_Protein/{dataset}/metadata.csv'
    res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

    # 读取 RNA 和 ATAC 数据
    adata_RNA, adata_ADT = read_RNA_ADT(rna_dir, adt_dir)

    # 处理 RNA 数据
    adata_RNA.layers["counts"] = adata_RNA.X.copy()
    sc.pp.normalize_total(adata_RNA)
    sc.pp.log1p(adata_RNA)
    sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=3000, subset=False)
    adata_RNA = adata_RNA[:, adata_RNA.var.highly_variable].copy()

    # 处理 ADT 数据
    adata_ADT.layers['counts'] = adata_ADT.X.copy()
    muon.prot.pp.clr(adata_ADT)
    adata_ADT.layers['clr'] = adata_ADT.X.copy()

    # 合并 RNA 和 ADT 数据
    adata = organize_multiome_anndatas(
        adatas=[[adata_RNA], [adata_ADT]],    # RNA-seq 始终在第一位
        layers=[['counts'], ['clr']]      # 使用 .layers 中的数据
    )

    # 读取元数据
    metadata = pd.read_csv(metadata_file, index_col=0)
    metadata.index = adata.obs_names  # 修正元数据的索引
    if dataset == '2_GSE128639':
        adata.obs['cell_type'] = metadata['celltype.l2'].astype('category')
    elif dataset == '7_zenodo6368128/LUNG':
        adata.obs['cell_type'] = metadata['celltype'].astype('category')
    elif dataset == '7_zenodo6368128/PBMC':
        adata.obs['cell_type'] = metadata['celltype'].astype('category')
    elif dataset == '12_GSE193181/P5':
        adata.obs['cell_type'] = metadata['ct_highres'].astype('category')
    elif dataset == '12_GSE193181/P8':
        adata.obs['cell_type'] = metadata['cell.type'].astype('category')

    # 处理缺失的 cell_type
    if adata.obs["cell_type"].isna().sum() != 0:
        adata.obs["cell_type"] = adata.obs["cell_type"].cat.add_categories(['NaN'])
        adata.obs.loc[adata.obs["cell_type"].isna(), "cell_type"] = 'NaN'

    result = pd.DataFrame()
    metrics_list = []
    method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']

    # 计算不同方法的指标
    for method in method_list:
        method_file = f'{res_dir}/{method}.csv'
        if os.path.exists(method_file):
            latent = pd.read_csv(method_file, header=None)
            latent.index = adata.obs_names
            adata.obsm[method] = latent
            print('the length of cell type:', adata.obs['cell_type'].nunique())
            method_cluster = method+'_cluster'
            if reso_dict is not None and dataset in reso_dict.keys():
                data_reso_dict = reso_dict[dataset]
                method_reso = data_reso_dict[method]
                sc.pp.neighbors(adata, use_rep=method, key_added=method)
                if method_reso is not None:
                    print(f'method_reso is not None, using the {method_reso} as resolution for {method}...')
                    sc.tl.leiden(adata,  random_state=0, key_added=method_cluster, 
                                 resolution=method_reso, neighbors_key=method)
                    sc.tl.umap(adata, neighbors_key=method)
                    print('Number of clusters:', adata.obs[method_cluster].nunique())
                else:
                    print('Searching resolution for {}...'.format(method))
                    supervised_clustering(adata, n_clusters=adata.obs['cell_type'].nunique(),
                                          key=method, add_key=method_cluster, cluster_method='leiden',
                                          start=start, end=end, increment=increment)
                    sc.tl.umap(adata, neighbors_key=method)
                    print('Number of clusters:', adata.obs[method_cluster].nunique())
            else:
                print('Searching resolution for {}...'.format(method))
                supervised_clustering(adata, n_clusters=adata.obs['cell_type'].nunique(),
                                      key=method, add_key=method_cluster, cluster_method='leiden',
                                      start=start, end=end, increment=increment)
                sc.tl.umap(adata, neighbors_key=method)
                print('Number of clusters:', adata.obs[method_cluster].nunique())
            # 计算指标
            # scib.metrics.cluster_optimal_resolution(adata, cluster_key="cluster", label_key="cell_type")
            ari = scib.metrics.ari(adata, cluster_key=method_cluster, label_key="cell_type")
            # iso_asw = scib.metrics.isolated_labels_asw(adata, label_key="cell_type", batch_key='batch', embed=method, verbose=False)
            nmi = scib.metrics.nmi(adata, cluster_key=method_cluster, label_key="cell_type")
            clisi = scib.metrics.clisi_graph(adata, label_key="cell_type", use_rep=method, type_='embed')
            sht = scib.metrics.silhouette(adata, label_key="cell_type", embed=method, scale=True) #  metric='euclidean',

            metrics_list.append([ari, nmi, clisi, sht, method])

    # 处理 Seurat 的结果 通过 graph 的方式
    if method == 'Seurat_graph':
        method = 'Seurat'
        con = mmread(f'{res_dir}/{method}_connectivities.mtx')
        dis = mmread(f'{res_dir}/{method}_distance.mtx')
        
        # uns['neighbors'] 的信息
        adata.uns['neighbors'] = {
            'connectivities_key': 'connectivities', 'distances_key': 'distances',
            'params': {'n_neighbors': 20, 'method': 'umap', 'random_state': 0, 'metric': 'euclidean'}
        }
        adata.uns['neighbors']['distances'] = csr_matrix(dis)
        adata.uns['neighbors']['connectivities'] = csr_matrix(con)
        adata.obsp['distances'] = csr_matrix(dis)
        adata.obsp['connectivities'] = csr_matrix(con)
        # Seurat 的信息   
        adata.uns['Seurat'] = {
            'connectivities_key': 'Seurat_connectivities', 'distances_key': 'Seurat_distances',
            'params': {'n_neighbors': 20, 'method': 'umap', 'random_state': 0, 'metric': 'euclidean', 'use_rep': 'Seurat'}
        }
        adata.uns['Seurat']['Seurat_distances'] = csr_matrix(dis)
        adata.uns['Seurat']['Seurat_connectivities'] = csr_matrix(con)
        adata.obsp['Seurat_distances'] = csr_matrix(dis)
        adata.obsp['Seurat_connectivities'] = csr_matrix(con)
        
        sc.tl.umap(adata) # , neighbors_key=method
        adata.obsm['Seurat'] = adata.obsm['X_umap'][:, :2]  # 确保只取二维的 UMAP 结果
        method_cluster = method+'_cluster'
        if reso_dict is not None and dataset in reso_dict.keys():
            data_reso_dict = reso_dict[dataset]
            method_reso = data_reso_dict[method]
            if method_reso is not None:
                print(f'method_reso is not None, using the {method_reso} as resolution for {method}...')
                sc.tl.leiden(adata,  random_state=0, key_added=method_cluster, 
                             resolution=method_reso, neighbors_key=method)
                print('Number of clusters:', adata.obs[method_cluster].nunique())
            else:
                print('Searching resolution for Seurat...')
                start = 0.3
                end = 1.5
                increment = 0.05
                for res in sorted(list(np.arange(start, end, increment)), reverse=True):
                   sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
                   count_unique = adata.obs['leiden'].nunique()
                   print('resolution={}, cluster number={}'.format(res, count_unique))
                   if count_unique == adata.obs['cell_type'].nunique():
                        break
                print('resolution for Seurat:', res)
                sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
                print('Number of clusters:', adata.obs['leiden'].nunique())
                adata.obs[method_cluster] = adata.obs['leiden']
        else:
            print('Searching resolution for Seurat...')
            start = 0.3
            end = 1.5
            increment = 0.05
            for res in sorted(list(np.arange(start, end, increment)), reverse=True):
               sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
               count_unique = adata.obs['leiden'].nunique()
               print('resolution={}, cluster number={}'.format(res, count_unique))
               if count_unique == adata.obs['cell_type'].nunique():
                    break
            print('resolution for Seurat:', res)
            sc.tl.leiden(adata, random_state=0, resolution=res, neighbors_key='Seurat')
            print('Number of clusters:', adata.obs['leiden'].nunique())
            adata.obs[method_cluster] = adata.obs['leiden']
    
        # 计算 Seurat 的指标
        # scib.metrics.cluster_optimal_resolution(adata, cluster_key="cluster", label_key="cell_type")
        ari = scib.metrics.ari(adata, cluster_key=method_cluster, label_key="cell_type")
        # iso_asw = scib.metrics.isolated_labels_asw(adata, label_key="cell_type", batch_key='batch', embed=method, verbose=False)
        nmi = scib.metrics.nmi(adata, cluster_key=method_cluster, label_key="cell_type")
        clisi = scib.metrics.clisi_graph(adata, label_key="cell_type", use_rep=method, type_='embed')
        sht = scib.metrics.silhouette(adata, label_key="cell_type", embed=method, scale=True) # metric='euclidean',
        metrics_list.append([ari, nmi, clisi, sht, 'Seurat'])

    # 保存结果
    df = pd.DataFrame(metrics_list, columns=['ARI', 'NMI', 'cLISI', 'cASW', 'model'])
    result = pd.concat([result, df], ignore_index=True)
    result['dataset'] = dataset
    if dataset == "7_zenodo6368128/LUNG" or dataset == "7_zenodo6368128/PBMC" or \
            dataset == "12_GSE193181/P5" or "12_GSE193181/P7":
        result.to_csv(f'{res_dir}/../../{dataset}_metrics_result.csv', index=False)
    else:
        result.to_csv(f'{res_dir}/../{dataset}_metrics_result.csv', index=False)
    print(dataset)

    return adata

In [5]:
def count_metrics_for_best_resolution(dataset):
    rna_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_Protein/{dataset}/RNA'
    adt_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_Protein/{dataset}'
    metadata_file = f'/data2/zhouwg_data/project/Garfield_benchmark/datasets/MultiomicsBenchmark/Raw_data/RNA_Protein/{dataset}/metadata.csv'
    res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

    # 读取 RNA 和 ATAC 数据
    adata_RNA, adata_ADT = read_RNA_ADT(rna_dir, adt_dir)

    # 处理 RNA 数据
    adata_RNA.layers["counts"] = adata_RNA.X.copy()
    sc.pp.normalize_total(adata_RNA)
    sc.pp.log1p(adata_RNA)
    sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=3000, subset=False)
    adata_RNA = adata_RNA[:, adata_RNA.var.highly_variable].copy()

    # 处理 ADT 数据
    adata_ADT.layers['counts'] = adata_ADT.X.copy()
    muon.prot.pp.clr(adata_ADT)
    adata_ADT.layers['clr'] = adata_ADT.X.copy()

    # 合并 RNA 和 ADT 数据
    adata = organize_multiome_anndatas(
        adatas=[[adata_RNA], [adata_ADT]],    # RNA-seq 始终在第一位
        layers=[['counts'], ['clr']]      # 使用 .layers 中的数据
    )

    # 读取元数据
    metadata = pd.read_csv(metadata_file, index_col=0)
    metadata.index = adata.obs_names  # 修正元数据的索引
    if dataset == '2_GSE128639':
        adata.obs['cell_type'] = metadata['celltype.l2'].astype('category')
    elif dataset == '7_zenodo6368128/LUNG':
        adata.obs['cell_type'] = metadata['celltype'].astype('category')
    elif dataset == '7_zenodo6368128/PBMC':
        adata.obs['cell_type'] = metadata['celltype'].astype('category')
    elif dataset == '12_GSE193181/P5':
        adata.obs['cell_type'] = metadata['ct_highres'].astype('category')
    elif dataset == '12_GSE193181/P8':
        adata.obs['cell_type'] = metadata['cell.type'].astype('category')

    # 处理缺失的 cell_type
    if adata.obs["cell_type"].isna().sum() != 0:
        adata.obs["cell_type"] = adata.obs["cell_type"].cat.add_categories(['NaN'])
        adata.obs.loc[adata.obs["cell_type"].isna(), "cell_type"] = 'NaN'

    adata.obs['batch'] = ['batch1'] * adata.shape[0]

    result = pd.DataFrame()
    metrics_list = []
    method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
    for method in method_list:
        method_file = f'{res_dir}/{method}.csv'
        if os.path.exists(method_file):
            latent = pd.read_csv(method_file, header=None)
            latent.index = adata.obs_names
            adata.obsm[method] = latent
            sc.pp.neighbors(adata, use_rep=method)
            sc.tl.umap(adata)
            # sc.tl.leiden(adata, key_added="cluster")
            scib.metrics.cluster_optimal_resolution(adata, cluster_key="cluster", label_key="cell_type")
            ari = scib.metrics.ari(adata, cluster_key="cluster", label_key="cell_type")
            # iso_asw = scib.metrics.isolated_labels_asw(adata, label_key="cell_type", batch_key='batch', embed=method,  verbose = False)
            nmi = scib.metrics.nmi(adata, cluster_key="cluster", label_key="cell_type")
            clisi = scib.metrics.clisi_graph(adata, label_key="cell_type",use_rep=method, type_='embed')
            sht = scib.metrics.silhouette(adata, label_key="cell_type", embed=method, metric='euclidean', scale=True)
            metrics_list.append([ari, nmi, clisi, sht, method])

    # 保存结果
    df = pd.DataFrame(metrics_list, columns=['ARI', 'NMI', 'cLISI', 'cASW', 'model'])
    result = pd.concat([result, df], ignore_index=True)
    result['dataset'] = dataset
    if dataset == "7_zenodo6368128/LUNG" or dataset == "7_zenodo6368128/PBMC" or \
            dataset == "12_GSE193181/P5" or "12_GSE193181/P7":
        result.to_csv(f'{res_dir}/../../{dataset}_metrics_result_best_reso.csv', index=False)
    else:
        result.to_csv(f'{res_dir}/../{dataset}_metrics_result_best_reso.csv', index=False)
        
    print(dataset)
    
    return adata

### 数据集 2_GSE128639

In [21]:
## 记录resolution
reso_dict = {
    '2_GSE128639': {
        'Garfield': 1.10,
        'Multigrate': 1.9000000000000012,
        'TotalVI': 1.8000000000000012,
        'scArches': 1.33,
        'Seurat': 0.15690529804500009, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('2_GSE128639', reso_dict=reso_dict)
adata

the length of cell type: 27
method_reso is not None, using the 1.1 as resolution for Garfield...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 1.9000000000000012 as resolution for Multigrate...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 1.8000000000000012 as resolution for TotalVI...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 1.33 as resolution for scArches...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 0.15690529804500009 as resolution for Seurat...
Number of clusters: 27
Chunk 501 does not have enough neighbors. Skipping...
Chunk 1885 does not have enough neighbors. Skipping...
Chunk 2996 does not have enough neighbors. Skipping...
Chunk 5648 does not have enough neighbors. Skipping...
Chunk 6042 does not have enough neighbors. Skipping...
Chunk 11226 does not have enough neighbors. Skipping...
Chunk 11366 does not hav

AnnData object with n_obs × n_vars = 30672 × 3025
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [22]:
import matplotlib.pyplot as plt

dataset = '2_GSE128639'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [23]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.526840,0.782347,0.999245,0.675484,Garfield,2_GSE128639
1,0.488701,0.754594,0.995517,0.566655,Multigrate,2_GSE128639
2,0.588347,0.798205,0.998372,0.568177,TotalVI,2_GSE128639
3,0.568930,0.793671,0.999415,0.598213,scArches,2_GSE128639
4,0.514337,0.800190,0.999907,0.721279,Seurat,2_GSE128639


#### 探索一下用最佳分辨率

In [25]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('2_GSE128639')
adata

resolution: 0.1, nmi: 0.8131184302239349
resolution: 0.2, nmi: 0.830460857130264
resolution: 0.3, nmi: 0.838938190188206
resolution: 0.4, nmi: 0.8609854022739019
resolution: 0.5, nmi: 0.8586232186212486
resolution: 0.6, nmi: 0.8408418206226269
resolution: 0.7, nmi: 0.8271416651336927
resolution: 0.8, nmi: 0.8138258028726302
resolution: 0.9, nmi: 0.7975184568962541
resolution: 1.0, nmi: 0.7994883442088038
resolution: 1.1, nmi: 0.7823473534830022
resolution: 1.2, nmi: 0.7870565047520184
resolution: 1.3, nmi: 0.7843099839005839
resolution: 1.4, nmi: 0.7771924177688436
resolution: 1.5, nmi: 0.7729157460809087
resolution: 1.6, nmi: 0.7762982872088157
resolution: 1.7, nmi: 0.7639290705720517
resolution: 1.8, nmi: 0.7488556205069352
resolution: 1.9, nmi: 0.7649305868181284
resolution: 2.0, nmi: 0.7496005105249844
optimised clustering against cell_type
optimal cluster resolution: 0.4
optimal score: 0.8609854022739019
resolution: 0.1, nmi: 0.7565831644467642
resolution: 0.2, nmi: 0.806635942628

AnnData object with n_obs × n_vars = 30672 × 3025
    obs: 'group', 'cell_type', 'batch', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [8]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '2_GSE128639'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.884222,0.860985,0.999245,0.675484,Garfield,2_GSE128639
1,0.835660,0.841850,0.995517,0.566655,Multigrate,2_GSE128639
2,0.877770,0.872664,0.998372,0.568177,TotalVI,2_GSE128639
3,0.872770,0.865969,0.999415,0.598213,scArches,2_GSE128639
4,0.663252,0.826611,0.999907,0.721279,Seurat,2_GSE128639


#### 利用最佳分辨率计算指标(Garfield);

In [6]:
## 记录resolution
reso_dict = {
    '2_GSE128639': {
        'Garfield': 0.4, # 1.10,
        'Multigrate': 1.9000000000000012,
        'TotalVI': 1.8000000000000012,
        'scArches': 1.33,
        'Seurat': 0.15690529804500009, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('2_GSE128639', reso_dict=reso_dict)
adata

the length of cell type: 27
method_reso is not None, using the 0.4 as resolution for Garfield...
Number of clusters: 17
the length of cell type: 27
method_reso is not None, using the 1.9000000000000012 as resolution for Multigrate...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 1.8000000000000012 as resolution for TotalVI...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 1.33 as resolution for scArches...
Number of clusters: 27
the length of cell type: 27
method_reso is not None, using the 0.15690529804500009 as resolution for Seurat...
Number of clusters: 27
Chunk 501 does not have enough neighbors. Skipping...
Chunk 1885 does not have enough neighbors. Skipping...
Chunk 2996 does not have enough neighbors. Skipping...
Chunk 5648 does not have enough neighbors. Skipping...
Chunk 6042 does not have enough neighbors. Skipping...
Chunk 11226 does not have enough neighbors. Skipping...
Chunk 11366 does not hav

AnnData object with n_obs × n_vars = 30672 × 3025
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [9]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '2_GSE128639'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.884222,0.860985,0.999245,0.675484,Garfield,2_GSE128639
1,0.488701,0.754594,0.995517,0.566655,Multigrate,2_GSE128639
2,0.588347,0.798205,0.998372,0.568177,TotalVI,2_GSE128639
3,0.568930,0.793671,0.999415,0.598213,scArches,2_GSE128639
4,0.514337,0.800190,0.999907,0.721279,Seurat,2_GSE128639


### 数据集 7_zenodo6368128/LUNG

In [17]:
## 记录resolution
reso_dict = {
    '7_zenodo6368128/LUNG': {
        'Garfield': 0.63,
        'Multigrate': 1.3000000000000007,
        'TotalVI': 0.8500000000000003,
        'scArches': 0.8000000000000003,
        'Seurat': 0.14121476824050008, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('7_zenodo6368128/LUNG', reso_dict=reso_dict)
adata

the length of cell type: 14
method_reso is not None, using the 0.63 as resolution for Garfield...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 1.3000000000000007 as resolution for Multigrate...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 0.8500000000000003 as resolution for TotalVI...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 0.8000000000000003 as resolution for scArches...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 0.14121476824050008 as resolution for Seurat...
Number of clusters: 14
Chunk 112 does not have enough neighbors. Skipping...
Chunk 125 does not have enough neighbors. Skipping...
Chunk 264 does not have enough neighbors. Skipping...
Chunk 265 does not have enough neighbors. Skipping...
Chunk 304 does not have enough neighbors. Skipping...
Chunk 357 does not have enough neighbors. Skipping...
Chunk 387 does 

AnnData object with n_obs × n_vars = 3362 × 3052
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [18]:
import matplotlib.pyplot as plt

dataset = '7_zenodo6368128/LUNG'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [19]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.502827,0.657247,0.968007,0.587787,Garfield,7_zenodo6368128/LUNG
1,0.493512,0.655153,0.970092,0.576290,Multigrate,7_zenodo6368128/LUNG
2,0.543276,0.683371,0.960239,0.518085,TotalVI,7_zenodo6368128/LUNG
3,0.508653,0.653584,0.964376,0.549389,scArches,7_zenodo6368128/LUNG
4,0.503213,0.720922,0.988793,0.609620,Seurat,7_zenodo6368128/LUNG


#### 探索一下用最佳分辨率

In [20]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('7_zenodo6368128/LUNG')
adata

resolution: 0.1, nmi: 0.6644695558697086
resolution: 0.2, nmi: 0.6814641468342751
resolution: 0.3, nmi: 0.681337585476829
resolution: 0.4, nmi: 0.6644673781401533
resolution: 0.5, nmi: 0.6564463148555076
resolution: 0.6, nmi: 0.6569589951078442
resolution: 0.7, nmi: 0.6486060000767506
resolution: 0.8, nmi: 0.638851076551211
resolution: 0.9, nmi: 0.6218751666499897
resolution: 1.0, nmi: 0.6199829310491367
resolution: 1.1, nmi: 0.6156395521040775
resolution: 1.2, nmi: 0.6031407467464517
resolution: 1.3, nmi: 0.6082796233390295
resolution: 1.4, nmi: 0.6071980881203342
resolution: 1.5, nmi: 0.600558774148804
resolution: 1.6, nmi: 0.596173986506624
resolution: 1.7, nmi: 0.5893560842811468
resolution: 1.8, nmi: 0.5886770668823079
resolution: 1.9, nmi: 0.589861313142517
resolution: 2.0, nmi: 0.5868339028666859
optimised clustering against cell_type
optimal cluster resolution: 0.2
optimal score: 0.6814641468342751
resolution: 0.1, nmi: 0.6499171479619003
resolution: 0.2, nmi: 0.704610937860476

AnnData object with n_obs × n_vars = 3362 × 3052
    obs: 'group', 'cell_type', 'batch', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [21]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '7_zenodo6368128/LUNG'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.565790,0.681464,0.968007,0.587787,Garfield,7_zenodo6368128/LUNG
1,0.638010,0.708285,0.970092,0.576290,Multigrate,7_zenodo6368128/LUNG
2,0.629156,0.731034,0.960239,0.518085,TotalVI,7_zenodo6368128/LUNG
3,0.587091,0.689774,0.964376,0.549389,scArches,7_zenodo6368128/LUNG
4,0.565291,0.735852,0.988793,0.609620,Seurat,7_zenodo6368128/LUNG


#### 利用最佳分辨率计算指标(Garfield);

In [22]:
## 记录resolution
reso_dict = {
    '7_zenodo6368128/LUNG': {
        'Garfield': 0.2,
        'Multigrate': 1.3000000000000007,
        'TotalVI': 0.8500000000000003,
        'scArches': 0.8000000000000003,
        'Seurat': 0.14121476824050008, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('7_zenodo6368128/LUNG', reso_dict=reso_dict)
adata

the length of cell type: 14
method_reso is not None, using the 0.2 as resolution for Garfield...
Number of clusters: 8
the length of cell type: 14
method_reso is not None, using the 1.3000000000000007 as resolution for Multigrate...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 0.8500000000000003 as resolution for TotalVI...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 0.8000000000000003 as resolution for scArches...
Number of clusters: 14
the length of cell type: 14
method_reso is not None, using the 0.14121476824050008 as resolution for Seurat...
Number of clusters: 14
Chunk 112 does not have enough neighbors. Skipping...
Chunk 125 does not have enough neighbors. Skipping...
Chunk 264 does not have enough neighbors. Skipping...
Chunk 265 does not have enough neighbors. Skipping...
Chunk 304 does not have enough neighbors. Skipping...
Chunk 357 does not have enough neighbors. Skipping...
Chunk 387 does no

AnnData object with n_obs × n_vars = 3362 × 3052
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [24]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '7_zenodo6368128/LUNG'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.565790,0.681464,0.968007,0.587787,Garfield,7_zenodo6368128/LUNG
1,0.493512,0.655153,0.970092,0.576290,Multigrate,7_zenodo6368128/LUNG
2,0.543276,0.683371,0.960239,0.518085,TotalVI,7_zenodo6368128/LUNG
3,0.508653,0.653584,0.964376,0.549389,scArches,7_zenodo6368128/LUNG
4,0.503213,0.720922,0.988793,0.609620,Seurat,7_zenodo6368128/LUNG


### 数据集 7_zenodo6368128/PBMC

In [31]:
## 记录resolution
reso_dict = {
    '7_zenodo6368128/PBMC': {
        'Garfield': 0.7500000000000002,
        'Multigrate': 0.8500000000000003,
        'TotalVI': 0.84,
        'scArches': 0.5,
        'Seurat': 0.098, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('7_zenodo6368128/PBMC', reso_dict=reso_dict)
adata

the length of cell type: 11
method_reso is not None, using the 0.7500000000000002 as resolution for Garfield...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.8500000000000003 as resolution for Multigrate...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.84 as resolution for TotalVI...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.5 as resolution for scArches...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.098 as resolution for Seurat...
Number of clusters: 11
Chunk 178 does not have enough neighbors. Skipping...
Chunk 245 does not have enough neighbors. Skipping...
Chunk 405 does not have enough neighbors. Skipping...
Chunk 550 does not have enough neighbors. Skipping...
Chunk 563 does not have enough neighbors. Skipping...
Chunk 573 does not have enough neighbors. Skipping...
Chunk 873 does not have enough neighbors. Sk

AnnData object with n_obs × n_vars = 7108 × 3052
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [32]:
import matplotlib.pyplot as plt

dataset = '7_zenodo6368128/PBMC'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [33]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.410132,0.640581,0.972405,0.588073,Garfield,7_zenodo6368128/PBMC
1,0.415778,0.617059,0.940645,0.533962,Multigrate,7_zenodo6368128/PBMC
2,0.467543,0.668589,0.926176,0.523700,TotalVI,7_zenodo6368128/PBMC
3,0.435581,0.591365,0.944616,0.522045,scArches,7_zenodo6368128/PBMC
4,0.422766,0.667751,0.984696,0.630589,Seurat,7_zenodo6368128/PBMC


#### 探索一下用最佳分辨率

In [34]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('7_zenodo6368128/PBMC')
adata

resolution: 0.1, nmi: 0.7045669196987505
resolution: 0.2, nmi: 0.7040936405475081
resolution: 0.3, nmi: 0.7205885851480783
resolution: 0.4, nmi: 0.7104931369492323
resolution: 0.5, nmi: 0.6709402505709019
resolution: 0.6, nmi: 0.6713295133715153
resolution: 0.7, nmi: 0.6540703032203694
resolution: 0.8, nmi: 0.6470365429842112
resolution: 0.9, nmi: 0.6420635132291619
resolution: 1.0, nmi: 0.6362029863885618
resolution: 1.1, nmi: 0.6342888505636097
resolution: 1.2, nmi: 0.6154469403548659
resolution: 1.3, nmi: 0.5982622762585442
resolution: 1.4, nmi: 0.6105697546713903
resolution: 1.5, nmi: 0.6029148420014858
resolution: 1.6, nmi: 0.6061772811323619
resolution: 1.7, nmi: 0.6005563366163184
resolution: 1.8, nmi: 0.5933744985630317
resolution: 1.9, nmi: 0.6025190440512225
resolution: 2.0, nmi: 0.5936765348885282
optimised clustering against cell_type
optimal cluster resolution: 0.3
optimal score: 0.7205885851480783
resolution: 0.1, nmi: 0.6509998007150417
resolution: 0.2, nmi: 0.6680341874

AnnData object with n_obs × n_vars = 7108 × 3052
    obs: 'group', 'cell_type', 'batch', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [35]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '7_zenodo6368128/PBMC'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.674712,0.720589,0.972405,0.588073,Garfield,7_zenodo6368128/PBMC
1,0.562749,0.690335,0.940645,0.533962,Multigrate,7_zenodo6368128/PBMC
2,0.639528,0.718717,0.926176,0.523700,TotalVI,7_zenodo6368128/PBMC
3,0.563754,0.666662,0.944616,0.522045,scArches,7_zenodo6368128/PBMC
4,0.425793,0.671251,0.984696,0.630589,Seurat,7_zenodo6368128/PBMC


#### 利用最佳分辨率计算指标(Garfield);

In [36]:
## 记录resolution
reso_dict = {
    '7_zenodo6368128/PBMC': {
        'Garfield': 0.3, # 0.7500000000000002,
        'Multigrate': 0.8500000000000003,
        'TotalVI': 0.84,
        'scArches': 0.5,
        'Seurat': 0.098, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('7_zenodo6368128/PBMC', reso_dict=reso_dict)
adata

the length of cell type: 11
method_reso is not None, using the 0.3 as resolution for Garfield...
Number of clusters: 8
the length of cell type: 11
method_reso is not None, using the 0.8500000000000003 as resolution for Multigrate...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.84 as resolution for TotalVI...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.5 as resolution for scArches...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.098 as resolution for Seurat...
Number of clusters: 11
Chunk 178 does not have enough neighbors. Skipping...
Chunk 245 does not have enough neighbors. Skipping...
Chunk 405 does not have enough neighbors. Skipping...
Chunk 550 does not have enough neighbors. Skipping...
Chunk 563 does not have enough neighbors. Skipping...
Chunk 573 does not have enough neighbors. Skipping...
Chunk 873 does not have enough neighbors. Skipping...
Chunk 

AnnData object with n_obs × n_vars = 7108 × 3052
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [37]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '7_zenodo6368128/PBMC'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.674712,0.720589,0.972405,0.588073,Garfield,7_zenodo6368128/PBMC
1,0.415778,0.617059,0.940645,0.533962,Multigrate,7_zenodo6368128/PBMC
2,0.467543,0.668589,0.926176,0.523700,TotalVI,7_zenodo6368128/PBMC
3,0.435581,0.591365,0.944616,0.522045,scArches,7_zenodo6368128/PBMC
4,0.422766,0.667751,0.984696,0.630589,Seurat,7_zenodo6368128/PBMC


### 数据集 12_GSE193181/P5

In [6]:
## 记录resolution
reso_dict = {
    '12_GSE193181/P5': {
        'Garfield': 0.70,
        'Multigrate': 0.83,
        'TotalVI': 0.6,
        'scArches': 0.58,
        'Seurat': 0.07, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('12_GSE193181/P5', reso_dict=reso_dict)
adata

the length of cell type: 15
method_reso is not None, using the 0.7 as resolution for Garfield...
Number of clusters: 15
the length of cell type: 15
method_reso is not None, using the 0.83 as resolution for Multigrate...
Number of clusters: 15
the length of cell type: 15
method_reso is not None, using the 0.6 as resolution for TotalVI...
Number of clusters: 15
the length of cell type: 15
method_reso is not None, using the 0.58 as resolution for scArches...
Number of clusters: 15
the length of cell type: 15
method_reso is not None, using the 0.07 as resolution for Seurat...
Number of clusters: 16
12_GSE193181/P5


AnnData object with n_obs × n_vars = 78530 × 3021
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [7]:
import matplotlib.pyplot as plt

dataset = '12_GSE193181/P5'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [9]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.283427,0.384462,0.898730,0.501129,Garfield,12_GSE193181/P5
1,0.112307,0.246496,0.838104,0.494953,Multigrate,12_GSE193181/P5
2,0.092961,0.213891,0.878303,0.505427,TotalVI,12_GSE193181/P5
3,0.117282,0.240685,0.864057,0.503540,scArches,12_GSE193181/P5
4,0.195190,0.356681,0.892815,0.458184,Seurat,12_GSE193181/P5


#### 探索一下用最佳分辨率

In [34]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('12_GSE193181/P5')
adata

resolution: 0.1, nmi: 0.7045669196987505
resolution: 0.2, nmi: 0.7040936405475081
resolution: 0.3, nmi: 0.7205885851480783
resolution: 0.4, nmi: 0.7104931369492323
resolution: 0.5, nmi: 0.6709402505709019
resolution: 0.6, nmi: 0.6713295133715153
resolution: 0.7, nmi: 0.6540703032203694
resolution: 0.8, nmi: 0.6470365429842112
resolution: 0.9, nmi: 0.6420635132291619
resolution: 1.0, nmi: 0.6362029863885618
resolution: 1.1, nmi: 0.6342888505636097
resolution: 1.2, nmi: 0.6154469403548659
resolution: 1.3, nmi: 0.5982622762585442
resolution: 1.4, nmi: 0.6105697546713903
resolution: 1.5, nmi: 0.6029148420014858
resolution: 1.6, nmi: 0.6061772811323619
resolution: 1.7, nmi: 0.6005563366163184
resolution: 1.8, nmi: 0.5933744985630317
resolution: 1.9, nmi: 0.6025190440512225
resolution: 2.0, nmi: 0.5936765348885282
optimised clustering against cell_type
optimal cluster resolution: 0.3
optimal score: 0.7205885851480783
resolution: 0.1, nmi: 0.6509998007150417
resolution: 0.2, nmi: 0.6680341874

AnnData object with n_obs × n_vars = 7108 × 3052
    obs: 'group', 'cell_type', 'batch', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [10]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '12_GSE193181/P5'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

FileNotFoundError: [Errno 2] No such file or directory: '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/12_GSE193181/P5_metrics_result_best_reso.csv'

#### 利用最佳分辨率计算指标(Garfield);

In [36]:
## 记录resolution
reso_dict = {
    '12_GSE193181/P5': {
        'Garfield': 0.70,
        'Multigrate': 0.83,
        'TotalVI': 0.6,
        'scArches': 0.58,
        'Seurat': 0.065, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('12_GSE193181/P5', reso_dict=reso_dict)
adata

the length of cell type: 11
method_reso is not None, using the 0.3 as resolution for Garfield...
Number of clusters: 8
the length of cell type: 11
method_reso is not None, using the 0.8500000000000003 as resolution for Multigrate...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.84 as resolution for TotalVI...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.5 as resolution for scArches...
Number of clusters: 11
the length of cell type: 11
method_reso is not None, using the 0.098 as resolution for Seurat...
Number of clusters: 11
Chunk 178 does not have enough neighbors. Skipping...
Chunk 245 does not have enough neighbors. Skipping...
Chunk 405 does not have enough neighbors. Skipping...
Chunk 550 does not have enough neighbors. Skipping...
Chunk 563 does not have enough neighbors. Skipping...
Chunk 573 does not have enough neighbors. Skipping...
Chunk 873 does not have enough neighbors. Skipping...
Chunk 

AnnData object with n_obs × n_vars = 7108 × 3052
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [37]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '12_GSE193181/P5'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.674712,0.720589,0.972405,0.588073,Garfield,7_zenodo6368128/PBMC
1,0.415778,0.617059,0.940645,0.533962,Multigrate,7_zenodo6368128/PBMC
2,0.467543,0.668589,0.926176,0.523700,TotalVI,7_zenodo6368128/PBMC
3,0.435581,0.591365,0.944616,0.522045,scArches,7_zenodo6368128/PBMC
4,0.422766,0.667751,0.984696,0.630589,Seurat,7_zenodo6368128/PBMC


### 数据集 12_GSE193181/P8

In [7]:
## 记录resolution
reso_dict = {
    '12_GSE193181/P8': {
        'Garfield': 0.38,
        'Multigrate': 0.6500000000000001,
        'TotalVI': 0.23914845000000007,
        'scArches': 0.2657205000000001,
        'Seurat': 0.05470949456575622, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('12_GSE193181/P8', reso_dict=reso_dict)
adata

the length of cell type: 10
method_reso is not None, using the 0.38 as resolution for Garfield...
Number of clusters: 10
the length of cell type: 10
Searching resolution for Multigrate...
Searching resolution...
resolution=1.9500000000000013, cluster number=28
resolution=1.9000000000000012, cluster number=28
resolution=1.8500000000000012, cluster number=27
resolution=1.8000000000000012, cluster number=26
resolution=1.750000000000001, cluster number=25
resolution=1.700000000000001, cluster number=25
resolution=1.650000000000001, cluster number=24
resolution=1.600000000000001, cluster number=22
resolution=1.550000000000001, cluster number=21
resolution=1.5000000000000009, cluster number=21
resolution=1.4500000000000008, cluster number=21
resolution=1.4000000000000008, cluster number=20
resolution=1.3500000000000008, cluster number=19
resolution=1.3000000000000007, cluster number=19
resolution=1.2500000000000007, cluster number=17
resolution=1.2000000000000006, cluster number=17
resolutio

AnnData object with n_obs × n_vars = 21042 × 3021
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [8]:
import matplotlib.pyplot as plt

dataset = '12_GSE193181/P8'
res_dir = f'/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein/{dataset}'

latent_key_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
for latent_key in latent_key_list:
    sc.tl.umap(adata, neighbors_key=latent_key)
    latent_key_cluster = latent_key + '_cluster'
    
    # Create a new figure for each plot with an appropriate size
    plt.figure(figsize=(12, 6))  # Adjust the figsize as needed
    sc.pl.umap(adata, color=['cell_type', latent_key_cluster], wspace=0.5, hspace=0.5, show=False, ncols=2)
    # Save the figure
    plt.savefig(res_dir + f"/{latent_key}_UMAP_clusters.png", bbox_inches='tight')  # Use bbox_inches to ensure full legend display
    plt.close()

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

<Figure size 2400x1200 with 0 Axes>

In [9]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.358035,0.441918,0.921081,0.514515,Garfield,12_GSE193181/P8
1,0.417393,0.443053,0.900223,0.506198,Multigrate,12_GSE193181/P8
2,0.178194,0.306676,0.898623,0.503576,TotalVI,12_GSE193181/P8
3,0.195646,0.286759,0.892970,0.501884,scArches,12_GSE193181/P8
4,0.300844,0.455369,0.936783,0.499165,Seurat,12_GSE193181/P8


#### 探索一下用最佳分辨率

In [10]:
# calculate the metrics based on the best resolution
adata = count_metrics_for_best_resolution('12_GSE193181/P8')
adata

resolution: 0.1, nmi: 0.5237007962606849
resolution: 0.2, nmi: 0.4857880065501947
resolution: 0.3, nmi: 0.4509984164183313
resolution: 0.4, nmi: 0.44145549926749994
resolution: 0.5, nmi: 0.4284877635697771
resolution: 0.6, nmi: 0.423577007776728
resolution: 0.7, nmi: 0.4127243580789494
resolution: 0.8, nmi: 0.43412165869081953
resolution: 0.9, nmi: 0.40740888591898494
resolution: 1.0, nmi: 0.38894606831311657
resolution: 1.1, nmi: 0.4045944231435975
resolution: 1.2, nmi: 0.39175197911229726
resolution: 1.3, nmi: 0.39014021071841265
resolution: 1.4, nmi: 0.3914893076208446
resolution: 1.5, nmi: 0.3885419600088378
resolution: 1.6, nmi: 0.3878478948785048
resolution: 1.7, nmi: 0.38880104468062987
resolution: 1.8, nmi: 0.3861920080182795
resolution: 1.9, nmi: 0.38792471264539635
resolution: 2.0, nmi: 0.3838597477670859
optimised clustering against cell_type
optimal cluster resolution: 0.1
optimal score: 0.5237007962606849
resolution: 0.1, nmi: 0.5114466792229645
resolution: 0.2, nmi: 0.504

AnnData object with n_obs × n_vars = 21042 × 3021
    obs: 'group', 'cell_type', 'batch', 'cluster'
    var: 'modality'
    uns: 'modality_lengths', 'neighbors', 'umap', 'cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'distances', 'connectivities'

In [11]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '12_GSE193181/P8'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result_best_reso.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.419280,0.523701,0.921081,0.514515,Garfield,12_GSE193181/P8
1,0.411968,0.511447,0.900223,0.506198,Multigrate,12_GSE193181/P8
2,0.154174,0.347789,0.898623,0.503576,TotalVI,12_GSE193181/P8
3,0.143288,0.338629,0.892970,0.501884,scArches,12_GSE193181/P8
4,0.268088,0.448721,0.936783,0.499165,Seurat,12_GSE193181/P8


#### 利用最佳分辨率计算指标(Garfield);

In [12]:
## 记录resolution
reso_dict = {
    '12_GSE193181/P8': {
        'Garfield': 0.1, #0.38,
        'Multigrate': 0.6500000000000001,
        'TotalVI': 0.23914845000000007,
        'scArches': 0.2657205000000001,
        'Seurat': 0.05470949456575622, 
    }
}

# search the resolution for each method
adata = count_metrics_for_resolution('12_GSE193181/P8', reso_dict=reso_dict)
adata

the length of cell type: 10
method_reso is not None, using the 0.1 as resolution for Garfield...
Number of clusters: 4
the length of cell type: 10
method_reso is not None, using the 0.6500000000000001 as resolution for Multigrate...
Number of clusters: 10
the length of cell type: 10
method_reso is not None, using the 0.23914845000000007 as resolution for TotalVI...
Number of clusters: 10
the length of cell type: 10
method_reso is not None, using the 0.2657205000000001 as resolution for scArches...
Number of clusters: 10
the length of cell type: 10
method_reso is not None, using the 0.05470949456575622 as resolution for Seurat...
Number of clusters: 10
12_GSE193181/P8


AnnData object with n_obs × n_vars = 21042 × 3021
    obs: 'group', 'cell_type', 'Garfield_cluster', 'Multigrate_cluster', 'TotalVI_cluster', 'scArches_cluster', 'Seurat_cluster'
    var: 'modality'
    uns: 'modality_lengths', 'Garfield', 'Garfield_cluster', 'umap', 'Multigrate', 'Multigrate_cluster', 'TotalVI', 'TotalVI_cluster', 'scArches', 'scArches_cluster', 'Seurat', 'Seurat_cluster'
    obsm: 'Garfield', 'X_umap', 'Multigrate', 'TotalVI', 'scArches', 'Seurat'
    layers: 'counts'
    obsp: 'Garfield_distances', 'Garfield_connectivities', 'Multigrate_distances', 'Multigrate_connectivities', 'TotalVI_distances', 'TotalVI_connectivities', 'scArches_distances', 'scArches_connectivities', 'Seurat_distances', 'Seurat_connectivities'

In [13]:
method_list = ['Garfield', 'Multigrate', 'TotalVI', 'scArches', 'Seurat']
save_path = '/data2/zhouwg_data/project/Garfield_benchmark/results/Vertical/RNA_Protein'
dataset = '12_GSE193181/P8'

df = pd.read_csv(f'{save_path}/{dataset}_metrics_result.csv')
# df = df.drop('Dataset', axis=0)
df
# PlotTable(df, num_embeds = len(method_list), min_max_scale=False, save_dir = save_path)

,ARI,NMI,cLISI,cASW,model,dataset
0,0.419280,0.523701,0.921081,0.514515,Garfield,12_GSE193181/P8
1,0.417393,0.443053,0.900223,0.506198,Multigrate,12_GSE193181/P8
2,0.178194,0.306676,0.898623,0.503576,TotalVI,12_GSE193181/P8
3,0.195646,0.286759,0.892970,0.501884,scArches,12_GSE193181/P8
4,0.300844,0.455369,0.936783,0.499165,Seurat,12_GSE193181/P8
